# 03_Map_par mostrar

En este notebook usamos `folium` para mostrar la rejilla espacial que generamos en `02_agrupar_por_lat_lon.ipynb`.

- Cada punto corresponde a un cuadro de `lat | long`.
- El popup muestra la `especie` dominante y su `cantidad`.
- `folium` es ideal para mapas interactivos ligeros y para embellecer la presentación.


In [ ]:
import os
import sys
import subprocess

try:
    import folium
    from folium.plugins import MarkerCluster
except ModuleNotFoundError:
    print("Folium no está instalado. Instalando folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])
    import folium
    from folium.plugins import MarkerCluster

print("folium version:", folium.__version__)


In [ ]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder     .appName("BiodiversidadColombiaFolium")     .master("local[*]")     .config("spark.driver.memory", "4g")     .getOrCreate()

parquet_path = os.path.join(os.getcwd(), "datos_para_mapa_rejilla.parquet")
print("Leyendo datos desde:", parquet_path)
df = spark.read.parquet(parquet_path)

print("Registros cargados:", df.count())
pandas_df = df.toPandas()
spark.stop()

pandas_df.head()


In [ ]:
import folium
from folium.plugins import MarkerCluster

if pandas_df.empty:
    raise ValueError("El DataFrame está vacío. Ejecuta primero la celda de carga de datos.")

center_lat = float(pandas_df["lat"].mean())
center_lon = float(pandas_df["long"].mean())
mapa = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="OpenStreetMap")
marker_cluster = MarkerCluster(name="Hotspots por cuadro").add_to(mapa)

def marker_color(cantidad):
    if cantidad >= 1000:
        return "darkred"
    if cantidad >= 500:
        return "red"
    if cantidad >= 100:
        return "orange"
    return "blue"

for _, row in pandas_df.iterrows():
    popup = folium.Popup(
        f"<b>Especie:</b> {row['especie']}<br>"
        f"<b>Cantidad:</b> {int(row['cantidad'])}<br>"
        f"<b>Lat:</b> {row['lat']}<br>"
        f"<b>Lon:</b> {row['long']}",
        max_width=300,
    )
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=max(4, min(16, int(row['cantidad'] / 100))),
        color=marker_color(int(row['cantidad'])),
        fill=True,
        fill_color=marker_color(int(row['cantidad'])),
        fill_opacity=0.7,
        popup=popup,
    ).add_to(marker_cluster)

title_html = '<h3 align="center">Mapa de cuadros de biodiversidad (Folium)</h3>'
mapa.get_root().html.add_child(folium.Element(title_html))

output_path = os.path.join(os.getcwd(), "03_map_cuadros_folium.html")
mapa.save(output_path)
print("Mapa guardado en:", output_path)
mapa


## ¿Qué puedes ajustar?

- Cambia el tamaño de los círculos en `radius=max(4, min(16, int(row["cantidad"] / 100)))` para que se vean mejor según tus datos.
- Cambia `round(..., 1)` en `02` por `round(..., 2)` para cuadros más pequeños.
- Si quieres etiquetas más grandes, puedes usar `folium.Marker` en lugar de `CircleMarker`.
